<a href="https://colab.research.google.com/github/herrypu/HP_LLM4CD_S6SP/blob/main/HW1/Docs/ChipChat_t0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Getting set up

## Setting up the notebook

In [26]:
### Installing dependencies
!pip install openai

!apt-get update
!apt-get install -y iverilog

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,632 B in 1s (3,793 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).

## Use LLM to generate Verilog code from natural language description

### Example 1: binary_to_bcd

In [27]:
! mkdir -p binary_to_bcd
! cd binary_to_bcd && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/binary_to_bcd_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1078  100  1078    0     0  10074      0 --:--:-- --:--:-- --:--:-- 10169


In [28]:
verilog_generation_prompt = '''
I am trying to create a Verilog model binary_to_bcd_converter for a binary to binary-coded-decimal converter. It must meet the following specifications:
	- Inputs:
		- Binary input (5-bits)
	- Outputs:
		- BCD (8-bits: 4-bits for the 10's place and 4-bits for the 1's place)

How would I write a design that meets these specifications?
'''

Now you can use the openai library to generate text from your prompt.

Before using LLMs for Verilog generation, you have to first get the api-key:

openai llms: https://platform.openai.com/api-keys
deepseek llms: https://platform.deepseek.com/sign_in
free llms provided by NVIDIA: https://build.nvidia.com/explore/discover

from openai import OpenAI
import os
client = OpenAI(
    api_key = "sk-9dfe3571281d46c1a801a2c7592a726b"
)

completion = client.chat.completions.create(
  model = "gpt-4o-mini",
  messages=[{"role":"user","content":verilog_generation_prompt}],
  max_tokens=1024, # limits the maximum number of tokens (words or pieces of words) that the model will generate in its response.
  stream=False
)

print(completion.choices[0].message.content)

In [29]:
from openai import OpenAI

# 1. 初始化客户端时，指向 DeepSeek 的服务器地址
client = OpenAI(
    api_key = "sk-2a1443da32bc49a1a4489e97b9b47367", # 替换为你的真实 Key
    base_url = "https://api.deepseek.com" # 必须添加这一行来重定向到 DeepSeek
)

# 2. 调用模型
completion = client.chat.completions.create(
  model = "deepseek-reasoner", # 这里建议使用 deepseek-chat (即 DeepSeek-V3)
  messages=[{"role": "user", "content": verilog_generation_prompt}],
  max_tokens=2048, # DeepSeek 支持更长的输出，建议设大一点防止代码截断
  temperature=0.7, # 适中的随机性，有助于生成更多样的设计逻辑
  stream=False
)

print(completion.choices[0].message.content)

Here's a Verilog model for a 5-bit binary to BCD converter:

```verilog
module binary_to_bcd_converter (
    input [4:0] binary_in,
    output reg [7:0] bcd_out
);

always @(*) begin
    // Initialize BCD output to 0
    bcd_out = 8'b0000_0000;
    
    // Double dabble algorithm implementation for 5-bit input
    // We'll handle values from 0-31
    if (binary_in > 29) begin
        bcd_out = {4'd3, (binary_in - 30)};
    end else if (binary_in > 19) begin
        bcd_out = {4'd2, (binary_in - 20)};
    end else if (binary_in > 9) begin
        bcd_out = {4'd1, (binary_in - 10)};
    end else begin
        bcd_out = {4'd0, binary_in[3:0]};
    end
end

endmodule
```

Alternative implementation using lookup table (more straightforward):

```verilog
module binary_to_bcd_converter (
    input [4:0] binary_in,
    output reg [7:0] bcd_out
);

always @(*) begin
    case(binary_in)
        5'd0: bcd_out = 8'h00;  // 00
        5'd1: bcd_out = 8'h01;  // 01
        5'd2: bcd_out = 8'h02;  //

Next, extract the generated verilog code from LLM response.

In [30]:
output_verilog_code = '''
module binary_to_bcd_converter (
    input wire [4:0] binary_input, // 5-bit binary input
    output reg [7:0] bcd_output     // 8-bit BCD output (4 bits for tens, 4 bits for ones)
);

// Internal registers for BCD representation
reg [3:0] bcd_tens; // Tens place
reg [3:0] bcd_ones; // Ones place
integer i; // Loop variable

always @* begin
    // Initialize BCD values to 0
    bcd_tens = 4'b0000;
    bcd_ones = 4'b0000;

    // Convert binary to BCD using double-dabble algorithm
    for (i = 0; i < 5; i = i + 1) begin
        // Shift left the BCD digits to make room for the next binary bit
        if (bcd_tens >= 5)
            bcd_tens = bcd_tens + 4'b0011; // Add 3 to tens place if necessary

        if (bcd_ones >= 5)
            bcd_ones = bcd_ones + 4'b0011; // Add 3 to ones place if necessary

        // Shift BCD values left to add a new binary bit
        {bcd_tens, bcd_ones} = {bcd_tens, bcd_ones} << 1;

        // Add the next bit of the binary input
        bcd_ones[0] = binary_input[4 - i]; // Add the next bit of the binary input
    end

    // Combine BCD places into the final output
    bcd_output = {bcd_tens, bcd_ones};
end

endmodule
'''
filename = "binary_to_bcd/binary_to_bcd.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

Use iverilog to verify the correctness of LLM generated verilog

In [31]:
!cd binary_to_bcd/ && iverilog -g2012 -o binary_to_bcd.vvp binary_to_bcd.v binary_to_bcd_tb.v && vvp binary_to_bcd.vvp

Testing Binary-to-BCD Converter...
VCD info: dumpfile my_design.vcd opened for output.
All test cases passed!


### Example 2: Sequence detector

In [32]:
! mkdir -p sequence_detector
! cd sequence_detector && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_detector_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2114  100  2114    0     0  27091      0 --:--:-- --:--:-- --:--:-- 27454


In [33]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a sequence detector. It must meet the following specifications:
	- Inputs:
		- Clock
		- Active-low reset
		- Data (3 bits)
	- Outputs:
		- Sequence found

While enabled, it should detect the following sequence of binary input values:
	- 0b001
	- 0b101
	- 0b110
	- 0b000
	- 0b110
	- 0b110
	- 0b011
	- 0b101

How would I write a design that meets these specifications?
'''

Next, extract the generated verilog code from LLM response.

In [34]:
output_verilog_code = '''
module sequence_detector (
    input         clk,
    input         reset_n,
    input  [2:0] data,
    output reg    sequence_found
);

    // State encoding
    typedef enum reg [3:0] {
        S_IDLE      = 4'b0000,
        S_001       = 4'b0001,
        S_101       = 4'b0010,
        S_110       = 4'b0011,
        S_000       = 4'b0100,
        S_011       = 4'b0101
    } state_t;

    // State variables
    state_t current_state, next_state;

    // Sequence found flag
    reg found_flag;

    // State transition
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            current_state <= S_IDLE;
        end else begin
            current_state <= next_state;
        end
    end

    // Next state logic
    always @(*) begin
        case (current_state)
            S_IDLE: begin
                if (data == 3'b001)
                    next_state = S_001;
                else
                    next_state = S_IDLE;
            end
            S_001: begin
                if (data == 3'b101)
                    next_state = S_101;
                else
                    next_state = S_IDLE;
            end
            S_101: begin
                if (data == 3'b110)
                    next_state = S_110;
                else
                    next_state = S_IDLE;
            end
            S_110: begin
                if (data == 3'b000)
                    next_state = S_000;
                else
                    next_state = S_IDLE;
            end
            S_000: begin
                if (data == 3'b110)
                    next_state = S_110;
                else
                    next_state = S_IDLE;
            end
            // Return to S_IDLE for any unrecognized states
            default: begin
                next_state = S_IDLE;
            end
        endcase
    end

    // Output logic
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            sequence_found <= 0;
        end else begin
            // Set found_flag only for successful transitions
            sequence_found <= (current_state == S_110 && next_state == S_011) ||
                              (current_state == S_011 && next_state == S_101);
        end
    end

endmodule
'''
filename = "sequence_detector/sequence_detector.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [35]:
!cd sequence_detector/ && iverilog -g2012 -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

Error: Cycle 8, Expected: 1, Got: 0


### Example 3: Shift Register

In [36]:
! mkdir -p shift_register
! cd shift_register && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/shift_register_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1367  100  1367    0     0  17326      0 --:--:-- --:--:-- --:--:-- 17525


In [37]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a shift register. It must meet the following specifications:
	- Inputs:
		- Clock
		- Active-low reset
		- Data (1 bit)
		- Shift enable
	- Outputs:
		- Data (8 bits)

How would I write a design that meets these specifications?
'''

In [38]:
output_verilog_code = '''
module shift_register (
  input wire clk,        // Clock signal
  input wire reset_n,    // Active-low reset signal
  input wire data_in,    // Data input
  input wire shift_enable,
  output reg [7:0] data_out
);

always @(posedge clk or negedge reset_n) begin
  if (!reset_n) begin
    // Active-low reset, so when reset_n is 0, reset data_out to 0
    data_out <= 8'b00000000;
  end else if (shift_enable) begin
    // Shift data on the rising edge of the clock when shift_enable is 1
    data_out <= {data_out[6:0], data_in};
  end
end

endmodule
'''
filename = "shift_register/shift_register.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [39]:
!cd sequence_detector/ && iverilog -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

sequence_detector.v:10: syntax error
sequence_detector.v:10: error: Invalid module instantiation
sequence_detector.v:20: syntax error
sequence_detector.v:20: error: Invalid module instantiation


### Example 4: Abro State Machine

In [40]:
! mkdir -p abro_state_machine
!cd abro_state_machine && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/abro_state_machine_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2762  100  2762    0     0  36429      0 --:--:-- --:--:-- --:--:-- 36826


In [41]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for an ABRO state machine. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - A
        - B
    - Outputs:
        - O
        - State

Other than the main output from ABRO machine, it should output the current state of the machine for use in verification.

The states for this state machine should be one-hot encoded.

How would I write a design that meets these specifications?
'''

In [42]:
output_verilog_code = '''
'''
filename = "abro_state_machine/abro_state_machine.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [43]:
!cd abro_state_machine/ && iverilog -o abro_state_machine.vvp abro_state_machine.v abro_state_machine_tb.v && vvp abro_state_machine.vvp

abro_state_machine_tb.v:12: error: Unknown module type: abro_state_machine
2 error(s) during elaboration.
*** These modules were missing:
        abro_state_machine referenced 1 times.
***


### Example 5: Dice Roller

In [44]:
!mkdir -p dice_roller
!cd dice_roller && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/dice_roller_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3416  100  3416    0     0  45056      0 --:--:-- --:--:-- --:--:-- 44947


In [45]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a simulated dice roller. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - Die select (2-bits)
        - Roll
    - Outputs:
        - Rolled number (up to 8-bits)

The design should simulate rolling either a 4-sided, 6-sided, 8-sided, or 20-sided die, based on the input die select. It should roll when the roll input goes high and output the random number based on the number of sides of the selected die.

How would I write a design that meets these specifications?
'''

In [46]:
from openai import OpenAI

client = OpenAI(
  api_key = ""
)

completion = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[{"role":"user","content":verilog_generation_prompt}],
  max_tokens=1024, # limits the maximum number of tokens (words or pieces of words) that the model will generate in its response.
  stream=False
)

print(completion.choices[0].message.content)

AuthenticationError: Error code: 401 - {'error': {'message': "You didn't provide an API key. You need to provide your API key in an Authorization header using Bearer auth (i.e. Authorization: Bearer YOUR_KEY), or as the password field (with blank username) if you're accessing the API from your browser and are prompted for a username and password. You can obtain an API key from https://platform.openai.com/account/api-keys.", 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [ ]:
output_verilog_code = '''
'''
filename = "dice_roller/dice_roller.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [ ]:
!cd dice_roller/ && iverilog -o dice_roller.vvp dice_roller.v dice_roller_tb.v && vvp dice_roller.vvp

### Example 6: LFSR

In [ ]:
!mkdir -p lfsr
!cd lfsr && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/lfsr_tb.v

In [ ]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for an LFSR. It must meet the following specifications:
	- Inputs:
		- Clock
        - Active-low reset
	- Outputs:
		- Data (8-bits)

The initial state should be 10001010, and the taps should be at locations 1, 4, 6, and 7.

How would I write a design that meets these specifications?
'''

In [ ]:
output_verilog_code = '''
'''
filename = "lfsr/lfsr.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [ ]:
!cd lfsr/ && iverilog -o lfsr.vvp lfsr.v lfsr_tb.v && vvp lfsr.vvp

### Example 7: Sequence Generator

In [ ]:
!mkdir -p sequence_generator
!cd sequence_generator && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_generator_tb.v

In [ ]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a sequence generator. It must meet the following specifications:
	- Inputs:
		- Clock
		- Active-low reset
		- Enable
	- Outputs:
		- Data (8 bits)

While enabled, it should generate an output sequence of the following hexadecimal values and then repeat:
	- 0xAF
	- 0xBC
	- 0xE2
	- 0x78
	- 0xFF
	- 0xE2
	- 0x0B
	- 0x8D

How would I write a design that meets these specifications?
'''

In [ ]:
output_verilog_code = '''
'''
filename = "sequence_generator/sequence_generator.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [ ]:
!cd sequence_generator/ && iverilog -o sequence_generator.vvp sequence_generator.v sequence_generator_tb.v && vvp sequence_generator.vvp

### Example 8: Traffic Light

In [ ]:
!mkdir -p traffic_light/
!cd traffic_light/ && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/traffic_light_tb.v

In [ ]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a traffic light state machine. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - Enable
    - Outputs:
        - Red
        - Yellow
        - Green

The state machine should reset to a red light, change from red to green after 32 clock cycles, change from green to yellow after 20 clock cycles, and then change from yellow to red after 7 clock cycles.

How would I write a design that meets these specifications?
'''

In [ ]:
output_verilog_code = '''
'''
filename = "traffic_light/traffic_light.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [ ]:
!cd traffic_light/ && iverilog -o traffic_light.vvp traffic_light.v traffic_light_tb.v && vvp traffic_light.vvp